# 05. Evaluation

This notebook introduces a simple evaluation harness for measuring extraction quality.
Running an agent is only half the work — you also need to know whether the results are correct.

## Why evaluation matters in DH
In digital humanities, extraction pipelines are often applied to hundreds or thousands of documents.
Without a way to measure quality, it is impossible to know whether a change to the agent's instructions
improved or degraded the results. Even a small gold standard gives you an objective signal.

## Concepts
- Gold-standard annotations
- Precision, recall, and F1
- Exact-match evaluation
- Comparing agent outputs to expected results
- Surfacing systematic errors

## Dataset
Six sample documents in `data/` are used. Gold-standard annotations cover the five readable documents;
the OCR table (`letter_06`) is intentionally excluded from evaluation to illustrate that not every
document type can be meaningfully scored with exact-match metrics.

## Colab data setup
If the notebook is opened in Google Colab without cloning the repo, upload `data.zip` from the
repository root to the current working directory using the Files panel, then rerun the setup cell below.

## Further reading
- Agents: https://openai.github.io/openai-agents-python/agents
- Structured output: https://openai.github.io/openai-agents-python/results/
- Tracing: https://openai.github.io/openai-agents-python/tracing/

In [ ]:
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool, set_tracing_export_api_key, trace

DEFAULT_MODEL = 'gpt-5.4-mini'

In [ ]:
candidate_dirs = [Path.cwd() / 'data', Path.cwd().parent / 'data']
data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    zip_candidates = [Path.cwd() / 'data.zip', Path.cwd().parent / 'data.zip']
    zip_path = next((path for path in zip_candidates if path.exists()), None)
    if zip_path is not None:
        import zipfile
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(zip_path.parent)
        data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    if 'google.colab' in sys.modules:
        print('Colab detected, but data/ is still missing.')
        print('Upload data.zip from the repository root into the Files panel, then rerun this cell.')
    else:
        raise FileNotFoundError('data/ folder not found. Clone the repo locally or place data.zip next to the notebook.')

load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd().parent / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or your environment before running this notebook.'
set_tracing_export_api_key(os.environ['OPENAI_API_KEY'])


def load_documents() -> list[dict[str, str]]:
    documents = []
    for index, path in enumerate(sorted(data_dir.glob('*.txt')), start=1):
        documents.append({
            'id': index,
            'filename': path.name,
            'text': path.read_text(encoding='utf-8').strip(),
        })
    if not documents:
        raise FileNotFoundError('No .txt files found in data/.')
    return documents

DOCUMENTS = load_documents()
for doc in DOCUMENTS:
    print(doc['id'], doc['filename'])

## Step 1: Define the gold standard

A gold standard is a manually verified set of correct answers. In this notebook we write them
directly as Python dicts. In a real project they would come from a spreadsheet, a labelling tool,
or a peer-reviewed annotation scheme.

Two rules for good gold-standard data in DH:
1. **Be specific about what counts as correct.** Does "4 April 1871" match "April 1871"? Decide in advance.
2. **Annotate only what the text explicitly states.** If the text is ambiguous, mark it as such rather than guessing.

Here we use exact match after lowercasing, which is a strict but transparent criterion.

In [ ]:
# Keyed by filename so the mapping is robust to document ordering.
GOLD: dict[str, dict[str, list[str]]] = {
    'letter_01_madrid_1871.txt': {
        'people': ['Maria Gomez'],
        'places': ['Madrid', 'Valencia'],
        'dates':  ['4 April 1871'],
    },
    'letter_02_seville_1894.txt': {
        'people': ['Antonio Ruiz'],
        'places': ['Seville'],
        'dates':  ['18 June 1894'],
    },
    # Letter 03 is an OCR fragment — approximate annotations to show how noise affects scores.
    'letter_03_ocr_fragment.txt': {
        'people': ['Maria Gomez'],
        'places': ['Madrid'],
        'dates':  ['1871'],
    },
    'letter_04_barcelona_1902.txt': {
        'people': ['A. Castelló', 'Miquel Font', 'Ramon Vidal', 'Eduard Domènech'],
        'places': ['Barcelona', 'Girona'],
        'dates':  ['12 March 1902'],
    },
    'letter_05_porto_1888.txt': {
        'people': ['Mr. Harrington', 'Eça de Queirós', 'Isabel Ferreira', 'Rodrigues'],
        'places': ['Porto', 'Lisbon', 'Braga'],
        'dates':  ['3 September 1888'],
    },
    # letter_06_ocr_table.txt is intentionally omitted: the OCR table format is too
    # ambiguous for meaningful exact-match evaluation. The scoring loop will print
    # "No gold standard for letter_06_ocr_table.txt, skipping." — that is expected.
}

GOLD

## Step 2: Evaluation metrics

We measure three standard information-retrieval metrics per field:

| Metric | Formula | Meaning |
|--------|---------|----------|
| **Precision** | TP / (TP + FP) | Of all extracted items, how many are correct? |
| **Recall** | TP / (TP + FN) | Of all correct items, how many were found? |
| **F1** | 2·P·R / (P + R) | Harmonic mean — balances precision and recall |

TP = true positives (extracted and correct), FP = false positives (extracted but wrong),
FN = false negatives (correct but not extracted).

Both predicted and gold values are lowercased and stripped before comparison, so
"Madrid" and "madrid" are treated as equal.

In [ ]:
def precision_recall_f1(predicted: list[str], gold: list[str]) -> dict:
    pred_set = {v.strip().lower() for v in predicted}
    gold_set = {v.strip().lower() for v in gold}
    tp = len(pred_set & gold_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        'precision': round(precision, 2),
        'recall':    round(recall, 2),
        'f1':        round(f1, 2),
        'tp': tp, 'fp': fp, 'fn': fn,
    }


@dataclass
class Extraction:
    people:   list[str] = field(default_factory=list)
    places:   list[str] = field(default_factory=list)
    dates:    list[str] = field(default_factory=list)
    evidence: list[str] = field(default_factory=list)


def evaluate(extraction: Extraction, gold: dict) -> pd.DataFrame:
    rows = []
    for field_name in ('people', 'places', 'dates'):
        predicted = getattr(extraction, field_name, [])
        gold_values = gold.get(field_name, [])
        metrics = precision_recall_f1(predicted, gold_values)
        rows.append({'field': field_name, **metrics})
    return pd.DataFrame(rows)


# Smoke-test with a perfect prediction.
mock = Extraction(people=['Maria Gomez'], places=['Madrid', 'Valencia'], dates=['4 April 1871'])
evaluate(mock, GOLD['letter_01_madrid_1871.txt'])

## Step 3: Define and run the agent

This is the same `archive_agent` from notebook 01. Running it here confirms the evaluation
harness works end-to-end: agent produces an extraction, we score it against gold.

In [ ]:
archive_agent = Agent(
    name='Archive Extractor',
    instructions=(
        'Extract people, places, dates, and evidence from the historical text. '
        'Be conservative: only extract what the text explicitly states. '
        'Include a short direct quote as evidence for each extracted item.'
    ),
    output_type=Extraction,
    model=DEFAULT_MODEL,
)

extractions: dict[str, Extraction] = {}

for doc in DOCUMENTS:
    with trace(f"evaluation: {doc['filename']}"):
        result = await Runner.run(archive_agent, doc['text'])
    extractions[doc['filename']] = result.final_output
    print(f"✓ {doc['filename']}")

extractions

## Step 4: Score each document

Compare the agent's output to the gold standard for every document and field.
Scan the table for patterns:
- Low recall on `dates` may mean the agent misses implicit date references.
- Low precision on `people` may mean it extracts role names ("Editor", "brother") as people.
- Poor scores on letter 03 are expected — OCR noise is the root cause.

In [ ]:
all_rows = []
for filename, extraction in extractions.items():
    gold = GOLD.get(filename)
    if gold is None:
        print(f'No gold standard for {filename}, skipping.')
        continue
    df = evaluate(extraction, gold)
    df.insert(0, 'document', filename)
    all_rows.append(df)

results = pd.concat(all_rows, ignore_index=True)
results

In [ ]:
# Macro-average across all documents per field.
summary = results.groupby('field')[['precision', 'recall', 'f1']].mean().round(2)
print('Macro-average across all documents:')
summary

## Discussion: sourcing gold standards in real DH projects

A three-document gold standard is enough to test a harness, but not enough to trust results.
Here are practical sources for larger gold standards:

| Source | Notes |
|--------|-------|
| **Manual annotation** | Annotate a stratified sample (clean + noisy + edge cases). Use at least two annotators and measure inter-annotator agreement. |
| **Existing DH datasets** | Projects like [CLEF HIPE](https://hipe-eval.github.io/HIPE-2022/) provide annotated historical NER corpora. |
| **Expert review of outputs** | Run the agent, have a domain expert correct the outputs, then treat the corrected results as gold. |
| **Synthetic generation** | Generate documents with known entities for unit-testing edge cases (OCR noise, name variants, ambiguous dates). |

**How many documents do you need?** For a quick sanity check, 10–20 manually annotated documents
are enough to detect systematic errors. For a publishable DH pipeline, aim for a sample that
covers the full distribution of document types and quality levels in your collection.

## Exercise

The current metric uses **exact match** after lowercasing. This means `"Maria Gomez"` and `"Gomez"` score as a miss even though one is a substring of the other — a common situation with historical name variants.

Extend `precision_recall_f1` to use **partial match**: a predicted value counts as a true positive if any gold value contains it as a substring (or vice versa).

Compare the partial-match scores to the exact-match scores on the same documents. Which field benefits most?

## Solution hint

Replace the set-intersection logic with a loop that checks `any(pred in g or g in pred for g in gold_set)`.